# Session 1 · First introduction to Python · SOLUTIONS

You will
1. explore a first table with **pandas**: the subjects of the Sleep-EDF dataset we use this week
2. learn the **numpy** basics for signals (arrays, 0-based indexing, slicing, masks)
3. make your first **matplotlib** figures.

**How to work**
- `Shift+Enter` runs a cell. Run the cells from top to bottom. The kernel (top right) must be `.venv`.
- Cells marked `# TODO` are yours: replace the `...` with code.
- Solutions are shared after the session.
- AI help (chat, inline completion) is allowed. Try first yourself, and make sure you understand every line you keep.

## Python vs. MATLAB vs. R

|  | MATLAB | R | Python (numpy / pandas) |
|---|---|---|---|
| First element | `x(1)` | `x[1]` | `x[0]` |
| Elements 1 to 3 | `x(1:3)` | `x[1:3]` | `x[0:3]` (end **excluded**) |
| Last element | `x(end)` | `x[length(x)]` | `x[-1]` |
| Element-wise product | `a .* b` | `a * b` | `a * b` |
| Matrix product | `a * b` | `a %*% b` | `a @ b` |
| Table | `table` | `data.frame` / tibble | `pd.DataFrame` |
| One column | `T.age` | `df$age` | `df["age"]` |
| Filter rows | `T(T.age > 65, :)` | `df[df$age > 65, ]` | `df[df["age"] > 65]` |
| Group summary | `groupsummary` | `group_by() \|> summarise()` | `df.groupby(...).agg(...)` |
| Help | `help mean` | `?mean` | `help(np.mean)` |

Two more differences: Python uses **indentation** (not `end`) to mark blocks, and packages are loaded with `import`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path("..") / "data"  # the notebook runs inside notebooks/, the data is one level up

## 1. A first table

`SC-subjects.csv` lists the people in the Sleep-EDF dataset. **One row = one recorded night.**

| Column | Meaning |
|---|---|
| `subject` | subject number (0–82) |
| `night` | 1 or 2 |
| `age` | age in years |
| `sex_F` | 1 = female, 2 = male |
| `LightsOff` | clock time when the lights went off (we use it later) |

In [ ]:
df = pd.read_csv(DATA_DIR / "SC-subjects.csv")
df.head()  # first 5 rows

In [ ]:
print(df.shape)  # (rows, columns)
df.info()        # column types and missing values

In [ ]:
df.describe()  # summary statistics of the numeric columns

### Exercise 1.1: nights vs. subjects

The table has 153 rows, but that doesn't mean 153 people.

In [ ]:
n_subjects = df["subject"].nunique()
print("Subjects:", n_subjects)

nights_per_subject = df["subject"].value_counts()

nights_per_subject.value_counts()

## 2. Selecting columns and rows

In [ ]:
ages = df["age"]       # one column -> a Series
df[["subject", "age"]] # several columns (note the double brackets) -> a DataFrame

In [ ]:
df.iloc[0:3]  # rows by POSITION: rows 0, 1 and 2 (the end, 3, is excluded)

In [ ]:
df.loc[df["age"] > 90]  # rows by CONDITION

⚠️ **Combining conditions:** use `&` (and), `|` (or), `~` (not), and put **parentheses** around each condition. Python's `and` / `or` don't work on columns.

In [ ]:
df[(df["age"] > 80) & (df["sex_F"] == 1)]  # women older than 80

### Exercise 2.1

In [ ]:
oldest = df["age"].max()
youngest = df["age"].min()
print("Oldest:", oldest, "| youngest:", youngest)

older = df[df["age"] >= 65]
print("Nights from subjects aged 65+:", len(older))

## 3. New columns, one row per subject, group summaries

Numbers like `1`/`2` for sex are easy to mix up. Let's add a readable column with `.map()`:

In [ ]:
df["sex"] = df["sex_F"].map({1: "F", 2: "M"})
df.head()

Age and sex are properties of the **subject**, not of the night. For questions about people, we keep one row per subject.

`.copy()` makes it explicit that `subjects` is a new table: changing it never changes `df`. (numpy behaves differently, see Section 4.)

In [ ]:
subjects = df.drop_duplicates("subject")[["subject", "age", "sex"]].copy()
print(len(subjects), "subjects")
subjects.head()

`pd.cut` sorts a number into groups (bins). Here: 4 age groups of similar size.

In [ ]:
bins = [0, 44, 64, 79, 120]
labels = ["25-44", "45-64", "65-79", "80+"]
subjects["age_group"] = pd.cut(subjects["age"], bins=bins, labels=labels)

# Group summary, like group_by() |> summarise() in R
subjects.groupby("age_group", observed=True)["age"].agg(["count", "mean", "min", "max"])

### Exercise 3.1

In [ ]:
counts = pd.crosstab(subjects["age_group"], subjects["sex"])
counts

In [ ]:
subjects.groupby("sex")["age"].mean()

## 4. numpy: arrays for signals

This afternoon, MNE gives you the EEG as a **numpy array of shape `(n_channels, n_samples)`**. Sleep-EDF is sampled at **100 Hz**, so 30 s of signal are 30 × 100 = 3000 samples.

To practise, we build a fake 2-channel epoch.

In [ ]:
sfreq = 100                      # sampling rate (Hz)
t = np.arange(0, 30, 1 / sfreq)  # time of each sample (s); the end, 30, is excluded
print(t.shape, t[:4], round(t[-1], 2))  # without round(): 29.990000000000002, a tiny rounding error of floating-point numbers

In [ ]:
rng = np.random.default_rng(seed=0)
alpha = 20 * np.sin(2 * np.pi * 10 * t)  # a 10 Hz "alpha" wave, 20 µV
noise = 5 * rng.standard_normal(t.size)  # random noise, about 5 µV

eeg = np.vstack([alpha + noise, noise])  # stack 2 "channels" as rows
eeg.shape                                # (n_channels, n_samples)

Indexing a 2-D array: `eeg[channel, sample]`

| Code | Meaning |
|---|---|
| `eeg[0]` | all samples of the first channel |
| `eeg[:, :100]` | the first second (100 samples) of all channels |
| `eeg[1, -1]` | the last sample of the second channel |
| `eeg * 1e-6` | every value × 10⁻⁶ (µV → V; MNE works in volts!) |
| `np.abs(eeg) > 20` | a boolean array: where is the amplitude above 20 µV? |

### Exercise 4.1

In [ ]:
first_2s = eeg[0, : 2 * sfreq]

idx = int(12.5 * sfreq)

peak = np.abs(eeg).max(axis=1)

print(first_2s.shape, idx, t[idx], peak)

### Exercise 4.2: boolean masks

In [ ]:
frac = np.mean(np.abs(eeg[0]) > 20)
print(f"{frac:.1%} of samples above 20 µV")

⚠️ **Views vs. copies (different from MATLAB and R!)** A slice of a numpy array is a *view* on the same data. Changing the slice changes the original:

In [ ]:
a = np.arange(5)
b = a[1:3]       # a view
b[0] = 99
print(a)         # a changed too!

c = a[1:3].copy()  # an independent copy
c[0] = -1
print(a)           # a is unchanged

> **Tip:** never type sample numbers by hand. Always compute them from `sfreq`, so the same code works for any sampling rate.

## 5. Plotting with matplotlib

The pattern: `fig, ax = plt.subplots()` → draw on `ax` → label the axes **with units** → `plt.show()`.

In [ ]:
n = 3 * sfreq  # 3 seconds
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t[:n], eeg[0, :n], label="channel 0 (alpha + noise)")
ax.plot(t[:n], eeg[1, :n], label="channel 1 (noise)")
ax.set(xlabel="Time (s)", ylabel="Amplitude (µV)", title="3 s of a fake EEG epoch")
ax.legend()
plt.show()

### Exercise 5.1: age distribution

In [ ]:
fig, ax = plt.subplots()
for sex in ["F", "M"]:
    ax.hist(subjects.loc[subjects["sex"] == sex, "age"], bins=range(20, 110, 5), alpha=0.5, label=sex)
ax.set(xlabel="Age (years)", ylabel="Number of subjects", title="Age of the Sleep-EDF subjects")
ax.legend()
plt.show()

## Optional exercises

1. **Write a function** `seconds_to_samples(seconds, sfreq)` that returns the sample index (an `int`) for a time in seconds. Check it with `t[seconds_to_samples(12.5, sfreq)]`.
2. **A two-panel figure:** put the signal plot and your age histogram side by side with `plt.subplots(1, 2, figsize=(10, 4))`, and save it with `fig.savefig("first_figure.png", dpi=150)`.

In [ ]:
# Optional exercise 1
def seconds_to_samples(seconds, sfreq):
    return int(round(seconds * sfreq))


print(seconds_to_samples(12.5, sfreq), t[seconds_to_samples(12.5, sfreq)])

In [ ]:
# Optional exercise 2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(t[:n], eeg[0, :n])
ax1.set(xlabel="Time (s)", ylabel="Amplitude (µV)", title="Fake EEG, channel 0")
for sex in ["F", "M"]:
    ax2.hist(subjects.loc[subjects["sex"] == sex, "age"], bins=range(20, 110, 5), alpha=0.5, label=sex)
ax2.set(xlabel="Age (years)", ylabel="Number of subjects", title="Age distribution")
ax2.legend()
fig.tight_layout()
fig.savefig("first_figure.png", dpi=150)
plt.show()